###  1. Defining the schema for the sprints input
Folder with multiple json(Multi line) files

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
%run ../00.Common/02.Helper_Notebook

In [0]:
source_file = f"{landing_folder_path}/sprints/"
table_name = f"{catalog_name}.{bronze_schema}.sprints"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, FloatType
sprints_schema = StructType([
    StructField('date', DateType()),
    StructField('raceName',StringType()),
    StructField('round', IntegerType()),
    StructField('season', IntegerType()),
    StructField('url',StringType()),
    StructField('constructorId',StringType()),
    StructField('driverId',StringType()),
    StructField('grid',IntegerType()),
    StructField('laps',IntegerType()),
    StructField('number',IntegerType()),
    StructField('points',FloatType()),
    StructField('position',IntegerType()),
    StructField('positionText',StringType()),
    StructField('status',StringType())
])

### 2. Reading the json files from the results folder

In [0]:
sprints_df = (
    spark.read
    .format('json')
    .option('mode','FAILFAST')
    .option('multiLine','true')
    .schema(sprints_schema)
    .load(source_file)
     )

### 3. View the results df

In [0]:
display(sprints_df)

### 4. Adding the file ingestion timestamp and sourcefile metadata in the df

In [0]:
final_sprints = add_timestamp_metadata(sprints_df)

### 5. Writing the final dataframe into the table under bronze schema

In [0]:
(
    final_sprints.write.format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))

In [0]:
%sql
select season, count(*) from formula1.bronze.sprints group by season order by season;